In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install alpaca-py

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# ════════════════════════════════════════════════════════════════
# DATA DOWNLOAD — Multi-asset 1h bars via Alpaca
# ════════════════════════════════════════════════════════════════
# For Colab: run setup_colab_secrets() if Alpaca keys are in Colab Secrets

TICKERS = ['QQQ', 'SPY', 'NVDA', 'AAPL', 'MSFT', 'AMD', 'META', 'AMZN']
PRIMARY_TICKER = 'QQQ'
TICKER = PRIMARY_TICKER  # For export cell compatibility
START_DATE = '2024-01-01'

all_data = download_multi(TICKERS, START_DATE, timeframe='1h')

print(f"\nDownloaded {len(all_data)} tickers")
for t, df in all_data.items():
    print(f"  {t}: {len(df)} bars, {df.index.min()} to {df.index.max()}")

# Primary ticker data for strategy development
hourly_df = all_data[PRIMARY_TICKER].copy()
print(f"\nPrimary: {PRIMARY_TICKER} — {len(hourly_df)} hourly bars")
hourly_df.head()

In [ ]:
# ════════════════════════════════════════════════════════════════
# BUILD DAILY SIGNAL DATAFRAME
# ════════════════════════════════════════════════════════════════
# For each trading day:
#   - prev_close: previous day's last bar close
#   - first_bar: first hourly bar of the day
#   - last_bar: last hourly bar of the day
#   - morning_return: (first_bar.Close - prev_close) / prev_close
#   - last_hour_return: (last_bar.Close - second_to_last.Close) / second_to_last.Close
# ════════════════════════════════════════════════════════════════

def build_daily_signals(hdf):
    """Build daily signal DataFrame from hourly bars.
    
    Alpaca 1h bars for US equities: regular hours ~14:30-21:00 UTC.
    We group by date and filter to regular-hours bars.
    """
    # Ensure datetime index
    hdf = hdf.copy()
    if not isinstance(hdf.index, pd.DatetimeIndex):
        hdf.index = pd.to_datetime(hdf.index)
    
    # Group bars by trading date
    hdf['trade_date'] = hdf.index.date
    
    daily_rows = []
    dates = sorted(hdf['trade_date'].unique())
    
    for i, d in enumerate(dates):
        day_bars = hdf[hdf['trade_date'] == d].sort_index()
        
        # Need at least 3 bars (first bar, second-to-last, last)
        if len(day_bars) < 3:
            continue
        
        # Previous day's close
        if i == 0:
            continue  # No previous day for first day
        prev_date = dates[i - 1]
        prev_day_bars = hdf[hdf['trade_date'] == prev_date].sort_index()
        if len(prev_day_bars) == 0:
            continue
        prev_close = prev_day_bars['Close'].iloc[-1]
        
        first_bar = day_bars.iloc[0]
        second_to_last_bar = day_bars.iloc[-2]
        last_bar = day_bars.iloc[-1]
        
        morning_return = (first_bar['Close'] - prev_close) / prev_close
        last_hour_return = (last_bar['Close'] - second_to_last_bar['Close']) / second_to_last_bar['Close']
        
        daily_rows.append({
            'date': d,
            'prev_close': prev_close,
            'first_bar_open': first_bar['Open'],
            'first_bar_close': first_bar['Close'],
            'first_bar_volume': first_bar['Volume'],
            'second_to_last_close': second_to_last_bar['Close'],
            'last_bar_open': last_bar['Open'],
            'last_bar_close': last_bar['Close'],
            'last_bar_volume': last_bar['Volume'],
            'morning_return': morning_return,
            'last_hour_return': last_hour_return,
            'n_bars': len(day_bars),
            'first_bar_ts': day_bars.index[0],
            'second_to_last_ts': day_bars.index[-2],
            'last_bar_ts': day_bars.index[-1],
        })
    
    daily_df = pd.DataFrame(daily_rows)
    daily_df['date'] = pd.to_datetime(daily_df['date'])
    daily_df.set_index('date', inplace=True)
    return daily_df


daily_signals = build_daily_signals(hourly_df)
print(f"Built daily signal DataFrame: {len(daily_signals)} trading days")
print(f"Date range: {daily_signals.index[0].date()} to {daily_signals.index[-1].date()}")
print(f"\nMorning return stats:")
print(daily_signals['morning_return'].describe())
print(f"\nLast-hour return stats:")
print(daily_signals['last_hour_return'].describe())

# Predictive correlation — the core academic finding
corr = daily_signals['morning_return'].corr(daily_signals['last_hour_return'])
print(f"\n{'='*60}")
print(f"CORRELATION: morning_return vs last_hour_return = {corr:.4f}")
print(f"{'='*60}")
print(f"(Gao et al. found positive correlation ~0.08-0.12 on SPY)")

# Scatter plot
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(daily_signals['morning_return'] * 100, daily_signals['last_hour_return'] * 100,
           alpha=0.4, s=15, color='#2563EB')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Morning Return (%)', fontsize=11)
ax.set_ylabel('Last Hour Return (%)', fontsize=11)
ax.set_title(f'{PRIMARY_TICKER}: Morning Return vs Last-Hour Return (corr={corr:.4f})', fontsize=13)
ax.grid(True, alpha=0.3)

# Regression line
slope, intercept, r_val, p_val, std_err = stats.linregress(
    daily_signals['morning_return'], daily_signals['last_hour_return'])
x_line = np.linspace(daily_signals['morning_return'].min(), daily_signals['morning_return'].max(), 100)
ax.plot(x_line * 100, (slope * x_line + intercept) * 100, color='red', linewidth=2,
        label=f'OLS: slope={slope:.4f}, p={p_val:.4f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

daily_signals.head(10)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PREPARE DATA — IS/OOS 60/40 Split
# ════════════════════════════════════════════════════════════════

TRAIN_RATIO = 0.60
split_idx = int(len(daily_signals) * TRAIN_RATIO)

train_daily = daily_signals.iloc[:split_idx].copy()
val_daily = daily_signals.iloc[split_idx:].copy()

print(f"IN-SAMPLE:  {len(train_daily)} days  |  {train_daily.index[0].date()} to {train_daily.index[-1].date()}")
print(f"OOS:        {len(val_daily)} days  |  {val_daily.index[0].date()} to {val_daily.index[-1].date()}")

# Also build full hourly close series for vectorbt portfolio
hourly_close = hourly_df['Close'].astype(float).copy()
hourly_close.name = 'price'

# Build daily close from last bar of each day (for FTMO Monte Carlo)
stock_data = daily_signals[['last_bar_close']].rename(columns={'last_bar_close': 'Close'}).copy()
stock_data.columns = pd.Index(['Close'])

print(f"\nHourly close series: {len(hourly_close)} bars")
print(f"Daily close series:  {len(stock_data)} bars")

# Intraday Momentum Strategy

## Academic Basis
**"Market Intraday Momentum"** by Gao, Han, Li & Zhou  
*Journal of Financial Economics*, 2018  

### Core Finding
The **first half-hour return** (from previous close to 10:00 AM) significantly predicts the **last half-hour return** (3:30-4:00 PM) on the same day. This predictability is driven by informed trading in the morning and late-day portfolio rebalancing.

### Our Adaptation (Hourly Bars)
Since we use **1-hour bars** from Alpaca, we adapt the strategy:
- **Morning signal**: Return from previous day's close to the end of the **first hourly bar** of the current day
- **Last-hour trade**: Enter at the start of the **last hourly bar**, exit at the close

### Trading Rules
- If `morning_return > threshold`: **Go LONG** in the last hour
- If `morning_return < -threshold` (long-short mode): **Go SHORT** in the last hour
- Otherwise: **Stay flat**

### Key Properties
- **1 trade per day** maximum
- **1 hour of market exposure** per trade
- **No lookahead bias**: morning return is observed hours before the last-hour trade
- **Extremely mechanical**: only parameters are `threshold` and `long_only`
- **Optional**: Volume filter — only trade if first-bar volume exceeds its 20-day average

### Parameters
| Parameter | Description | Range |
|-----------|-------------|-------|
| `threshold` | Minimum \|morning_return\| to trigger a trade | 0.0 to 0.01 |
| `long_only` | If True, only take long signals | True / False |
| `volume_filter` | Require above-average first-bar volume | True / False |
| `vol_filter_mult` | Volume threshold multiplier | 0.8 to 1.5 |

In [ ]:
# ════════════════════════════════════════════════════════════════
# SIGNAL ENGINE — Intraday Momentum
# ════════════════════════════════════════════════════════════════

def generate_intraday_momentum_signals(hdf, daily_df, threshold=0.0, long_only=True,
                                        volume_filter=False, vol_filter_mult=1.0):
    """
    Intraday Momentum: first hour predicts last hour.
    
    For each trading day:
    1. Compute morning_return = (first_bar_close - prev_day_close) / prev_day_close
    2. If morning_return > threshold: enter LONG at start of last hourly bar
    3. If morning_return < -threshold and not long_only: enter SHORT at start of last bar
    4. Exit at end of day (last bar close)
    
    For vectorbt: entry on second-to-last bar, exit on last bar.
    This means we hold for exactly 1 bar = 1 hour.
    
    Anti-lookahead: morning return is known after first bar closes,
    well before the last bar.
    
    Parameters
    ----------
    hdf : DataFrame
        Hourly OHLCV data with DatetimeIndex.
    daily_df : DataFrame
        Daily signal DataFrame from build_daily_signals().
    threshold : float
        Minimum |morning_return| to trade. Default 0.0 (trade every day).
    long_only : bool
        If True, only long entries. If False, also short on negative mornings.
    volume_filter : bool
        If True, only trade when first bar volume > vol_filter_mult * 20-day avg.
    vol_filter_mult : float
        Multiplier for volume filter threshold.
    
    Returns
    -------
    entries : pd.Series (bool, aligned to hourly index)
    exits : pd.Series (bool, aligned to hourly index)
    short_entries : pd.Series (bool) — only meaningful if not long_only
    short_exits : pd.Series (bool) — only meaningful if not long_only
    """
    idx = hdf.index
    entries = pd.Series(False, index=idx, dtype=bool)
    exits = pd.Series(False, index=idx, dtype=bool)
    short_entries = pd.Series(False, index=idx, dtype=bool)
    short_exits = pd.Series(False, index=idx, dtype=bool)
    
    # Compute rolling 20-day average of first-bar volume for volume filter
    if volume_filter:
        vol_avg = daily_df['first_bar_volume'].rolling(20, min_periods=5).mean()
    
    for i in range(len(daily_df)):
        row = daily_df.iloc[i]
        mr = row['morning_return']
        
        # Volume filter check
        if volume_filter:
            avg_vol = vol_avg.iloc[i]
            if pd.isna(avg_vol) or row['first_bar_volume'] < vol_filter_mult * avg_vol:
                continue
        
        second_to_last_ts = row['second_to_last_ts']
        last_ts = row['last_bar_ts']
        
        # Check timestamps exist in hourly index
        if second_to_last_ts not in idx or last_ts not in idx:
            continue
        
        # Long signal: positive morning return above threshold
        if mr > threshold:
            entries.loc[second_to_last_ts] = True
            exits.loc[last_ts] = True
        
        # Short signal: negative morning return below -threshold
        elif not long_only and mr < -threshold:
            short_entries.loc[second_to_last_ts] = True
            short_exits.loc[last_ts] = True
    
    return entries, exits, short_entries, short_exits


# Quick test with default params
e_test, x_test, se_test, sx_test = generate_intraday_momentum_signals(
    hourly_df, daily_signals, threshold=0.0, long_only=True)
print(f"Test run (threshold=0.0, long_only=True):")
print(f"  Long entries: {e_test.sum()}")
print(f"  Long exits:   {x_test.sum()}")
print(f"  Days in data: {len(daily_signals)}")
print(f"  Trade rate:   {e_test.sum() / len(daily_signals) * 100:.1f}% of days")

In [ ]:
# ════════════════════════════════════════════════════════════════
# PARAMETER RANGES
# ════════════════════════════════════════════════════════════════

threshold_range = [0.0, 0.001, 0.002, 0.005, 0.01]  # minimum |morning_return| to trade
long_only_range = [True, False]                       # long only vs long+short
vol_filter_range = [False, True]                      # volume filter on/off
vol_filter_mult_range = [0.8, 1.0, 1.5]              # volume threshold multiplier

# Count combos (vol_filter_mult only matters when vol_filter=True)
n_combos = 0
combo_list = []
for th in threshold_range:
    for lo in long_only_range:
        for vf in vol_filter_range:
            if vf:
                for vm in vol_filter_mult_range:
                    combo_list.append((th, lo, vf, vm))
                    n_combos += 1
            else:
                combo_list.append((th, lo, vf, 1.0))
                n_combos += 1

print(f"Parameter ranges:")
print(f"  threshold:       {threshold_range}")
print(f"  long_only:       {long_only_range}")
print(f"  volume_filter:   {vol_filter_range}")
print(f"  vol_filter_mult: {vol_filter_mult_range} (only when volume_filter=True)")
print(f"\nTotal combinations: {n_combos}")
print(f"\nFirst 10 combinations:")
for c in combo_list[:10]:
    print(f"  threshold={c[0]}, long_only={c[1]}, vol_filter={c[2]}, vol_mult={c[3]}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# INITIALIZE RESULTS COLLECTION
# ════════════════════════════════════════════════════════════════

grid_search_results = []

INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

# Metrics to collect
METRIC_NAMES = [
    'threshold', 'long_only', 'volume_filter', 'vol_filter_mult',
    'total_return', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown',
    'ann_return', 'ann_volatility', 'calmar_ratio',
    'total_trades', 'win_rate', 'profit_factor', 'expectancy',
    'avg_win', 'avg_loss', 'largest_win', 'largest_loss',
    'payoff_ratio', 'trades_per_year',
]

print(f"Collecting {len(METRIC_NAMES)} metrics per parameter combination")
print(f"Metrics: {METRIC_NAMES}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# GRID SEARCH — Training Data Only
# ════════════════════════════════════════════════════════════════

# Build training-period hourly data
train_dates = set(train_daily.index.date)
hourly_df_copy = hourly_df.copy()
hourly_df_copy['trade_date'] = hourly_df_copy.index.date
train_hourly = hourly_df_copy[hourly_df_copy['trade_date'].isin(train_dates)].drop(columns=['trade_date'])
train_close = train_hourly['Close'].astype(float).copy()
train_close.name = 'price'

# Also build val hourly
val_dates = set(val_daily.index.date)
val_hourly = hourly_df_copy[hourly_df_copy['trade_date'].isin(val_dates)].drop(columns=['trade_date'])
val_close = val_hourly['Close'].astype(float).copy()
val_close.name = 'price'

print(f"Training hourly bars: {len(train_close)}")
print(f"Validation hourly bars: {len(val_close)}")
print(f"\nRunning grid search over {len(combo_list)} combinations...\n")

for ci, (th, lo, vf, vm) in enumerate(combo_list):
    try:
        e, x, se, sx = generate_intraday_momentum_signals(
            train_hourly, train_daily, threshold=th, long_only=lo,
            volume_filter=vf, vol_filter_mult=vm)
        
        # Build portfolio — long side
        if lo:  # long only
            pf = vbt.Portfolio.from_signals(
                close=train_close, entries=e, exits=x,
                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
        else:  # long + short
            pf = vbt.Portfolio.from_signals(
                close=train_close, entries=e, exits=x,
                short_entries=se, short_exits=sx,
                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
        
        trades = pf.trades
        n_trades = len(trades)
        if n_trades < 5:
            continue
        
        tr = np.asarray(trades.returns.values if hasattr(trades.returns, 'values') else trades.returns).ravel()
        pos = tr[tr > 0]
        neg = tr[tr < 0]
        years = max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9)
        
        def safe(fn, default=None):
            try: return float(fn())
            except: return default
        
        row = {
            'threshold': th,
            'long_only': lo,
            'volume_filter': vf,
            'vol_filter_mult': vm,
            'total_return': safe(pf.total_return),
            'sharpe_ratio': safe(lambda: pf.sharpe_ratio(freq='h')),
            'sortino_ratio': safe(lambda: pf.sortino_ratio(freq='h')),
            'max_drawdown': safe(pf.max_drawdown),
            'ann_return': safe(lambda: pf.annualized_return(freq='h')),
            'ann_volatility': safe(lambda: pf.annualized_volatility(freq='h')),
            'calmar_ratio': None,
            'total_trades': n_trades,
            'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else None,
            'profit_factor': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else None,
            'expectancy': float(tr.mean()) if len(tr) > 0 else None,
            'avg_win': float(pos.mean()) if len(pos) > 0 else None,
            'avg_loss': float(neg.mean()) if len(neg) > 0 else None,
            'largest_win': float(pos.max()) if len(pos) > 0 else None,
            'largest_loss': float(neg.min()) if len(neg) > 0 else None,
            'payoff_ratio': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else None,
            'trades_per_year': n_trades / years,
        }
        ann_ret = row['ann_return']
        max_dd = row['max_drawdown']
        if ann_ret is not None and max_dd is not None and abs(max_dd) > 1e-9:
            row['calmar_ratio'] = ann_ret / abs(max_dd)
        
        grid_search_results.append(row)
        
    except Exception as ex:
        pass

results_df = pd.DataFrame(grid_search_results)
print(f"\nCompleted: {len(results_df)} valid combinations out of {len(combo_list)}")

if not results_df.empty:
    print(f"\n{'='*90}")
    print(f"TOP 10 BY SHARPE RATIO (IN-SAMPLE)")
    print(f"{'='*90}")
    top10 = results_df.nlargest(10, 'sharpe_ratio')
    display_cols = ['threshold', 'long_only', 'volume_filter', 'vol_filter_mult',
                    'sharpe_ratio', 'total_return', 'max_drawdown', 'total_trades', 'win_rate', 'profit_factor']
    print(top10[display_cols].to_string(index=False))
else:
    print("No valid results found.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# OOS VALIDATION — Top 5 + Equity Curves
# ════════════════════════════════════════════════════════════════

if results_df.empty:
    print("No results to validate.")
else:
    top5 = results_df.nlargest(5, 'sharpe_ratio')
    
    oos_results = []
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{PRIMARY_TICKER} — Intraday Momentum: Top 5 IS vs OOS', fontsize=14, fontweight='bold')
    
    for rank, (_, row) in enumerate(top5.iterrows()):
        th = row['threshold']
        lo = bool(row['long_only'])
        vf = bool(row['volume_filter'])
        vm = row['vol_filter_mult']
        
        # IS portfolio (already computed, but rebuild for equity curve)
        e_is, x_is, se_is, sx_is = generate_intraday_momentum_signals(
            train_hourly, train_daily, threshold=th, long_only=lo,
            volume_filter=vf, vol_filter_mult=vm)
        if lo:
            pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                               init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
        else:
            pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                               short_entries=se_is, short_exits=sx_is,
                                               init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
        
        # OOS portfolio
        e_oos, x_oos, se_oos, sx_oos = generate_intraday_momentum_signals(
            val_hourly, val_daily, threshold=th, long_only=lo,
            volume_filter=vf, vol_filter_mult=vm)
        if lo:
            pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
        else:
            pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                                short_entries=se_oos, short_exits=sx_oos,
                                                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
        
        def safe(fn, default=None):
            try: return float(fn())
            except: return default
        
        is_sharpe = safe(lambda: pf_is.sharpe_ratio(freq='h'))
        oos_sharpe = safe(lambda: pf_oos.sharpe_ratio(freq='h'))
        is_ret = safe(pf_is.total_return)
        oos_ret = safe(pf_oos.total_return)
        is_dd = safe(pf_is.max_drawdown)
        oos_dd = safe(pf_oos.max_drawdown)
        
        oos_results.append({
            'rank': rank + 1,
            'threshold': th, 'long_only': lo, 'vol_filter': vf, 'vol_mult': vm,
            'is_sharpe': is_sharpe, 'oos_sharpe': oos_sharpe,
            'is_return': is_ret, 'oos_return': oos_ret,
            'is_dd': is_dd, 'oos_dd': oos_dd,
            'is_trades': len(pf_is.trades), 'oos_trades': len(pf_oos.trades),
        })
        
        # Equity curve
        ax = axes.flat[rank]
        eq_is = pf_is.value()
        eq_oos = pf_oos.value()
        ax.plot(eq_is.index, eq_is.values, color='#2563EB', linewidth=1.5, label='IS')
        ax.plot(eq_oos.index, eq_oos.values, color='#EA580C', linewidth=1.5, label='OOS')
        ax.set_title(f'#{rank+1}: th={th}, lo={lo}, vf={vf}\n'
                     f'IS Sharpe={is_sharpe:.3f} | OOS={oos_sharpe:.3f}' if is_sharpe and oos_sharpe else f'#{rank+1}',
                     fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    
    # Hide unused subplot
    axes.flat[5].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    oos_df = pd.DataFrame(oos_results)
    print(f"\n{'='*100}")
    print(f"IS vs OOS COMPARISON — Top 5")
    print(f"{'='*100}")
    print(oos_df.to_string(index=False))

## Sensitivity Analysis

This strategy has very few parameters (threshold, long_only, volume_filter), making it naturally robust to overfitting. The sensitivity analysis below examines how Sharpe ratio changes as we vary each parameter around the best configuration.

Key expectations:
- **Threshold**: Should show gradual degradation — too high means too few trades, too low means noisy signals
- **Long-only vs Long-Short**: Academic paper found stronger predictability on the long side
- **Volume filter**: Should improve quality but reduce trade count

In [ ]:
# ════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS CHARTS
# ════════════════════════════════════════════════════════════════

if results_df.empty:
    print("No results for sensitivity analysis.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    base_th = best['threshold']
    base_lo = bool(best['long_only'])
    base_vf = bool(best['volume_filter'])
    base_vm = best['vol_filter_mult']
    base_sharpe = float(best['sharpe_ratio'])
    
    print(f"Base config: threshold={base_th}, long_only={base_lo}, vol_filter={base_vf}, vol_mult={base_vm}")
    print(f"Base IS Sharpe: {base_sharpe:.4f}")
    
    # Sensitivity: vary threshold (hold other params at base)
    th_sens = results_df[
        (results_df['long_only'] == base_lo) &
        (results_df['volume_filter'] == base_vf) &
        (results_df['vol_filter_mult'] == base_vm)
    ].sort_values('threshold').copy()
    
    # Sensitivity: vary long_only
    lo_sens = results_df[
        (results_df['threshold'] == base_th) &
        (results_df['volume_filter'] == base_vf) &
        (results_df['vol_filter_mult'] == base_vm)
    ].sort_values('long_only').copy()
    
    # Sensitivity: vary volume_filter (keep threshold, long_only at base)
    vf_sens = results_df[
        (results_df['threshold'] == base_th) &
        (results_df['long_only'] == base_lo)
    ].copy()
    # Create a combined label
    vf_sens['vf_label'] = vf_sens.apply(
        lambda r: 'No Filter' if not r['volume_filter'] else f"VF x{r['vol_filter_mult']}", axis=1)
    vf_sens = vf_sens.drop_duplicates('vf_label').sort_values('vf_label')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Sensitivity Analysis — Base Sharpe: {base_sharpe:.4f}', fontsize=14, fontweight='bold')
    
    # 1. Threshold sensitivity
    if len(th_sens) > 0:
        th_sens['sharpe_delta'] = (th_sens['sharpe_ratio'] - base_sharpe) / abs(base_sharpe) * 100
        colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                  for x in th_sens['sharpe_delta']]
        axes[0].bar(th_sens['threshold'].astype(str), th_sens['sharpe_delta'], color=colors,
                    edgecolor='black', linewidth=0.5)
        axes[0].axhline(0, color='black', linewidth=1.5)
        axes[0].set_xlabel('Threshold', fontsize=11, fontweight='bold')
        axes[0].set_ylabel('Sharpe Delta %', fontsize=11, fontweight='bold')
        axes[0].set_title('Threshold Sensitivity', fontsize=12, fontweight='bold')
        axes[0].grid(axis='y', alpha=0.3)
    
    # 2. Long-only sensitivity
    if len(lo_sens) > 0:
        lo_sens['sharpe_delta'] = (lo_sens['sharpe_ratio'] - base_sharpe) / abs(base_sharpe) * 100
        colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                  for x in lo_sens['sharpe_delta']]
        axes[1].bar(lo_sens['long_only'].astype(str), lo_sens['sharpe_delta'], color=colors,
                    edgecolor='black', linewidth=0.5)
        axes[1].axhline(0, color='black', linewidth=1.5)
        axes[1].set_xlabel('Long Only', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Sharpe Delta %', fontsize=11, fontweight='bold')
        axes[1].set_title('Long-Only vs Long-Short', fontsize=12, fontweight='bold')
        axes[1].grid(axis='y', alpha=0.3)
    
    # 3. Volume filter sensitivity
    if len(vf_sens) > 0:
        vf_sens['sharpe_delta'] = (vf_sens['sharpe_ratio'] - base_sharpe) / abs(base_sharpe) * 100
        colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                  for x in vf_sens['sharpe_delta']]
        axes[2].bar(vf_sens['vf_label'], vf_sens['sharpe_delta'], color=colors,
                    edgecolor='black', linewidth=0.5)
        axes[2].axhline(0, color='black', linewidth=1.5)
        axes[2].set_xlabel('Volume Filter', fontsize=11, fontweight='bold')
        axes[2].set_ylabel('Sharpe Delta %', fontsize=11, fontweight='bold')
        axes[2].set_title('Volume Filter Sensitivity', fontsize=12, fontweight='bold')
        axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print(f"\nSENSITIVITY SUMMARY")
    print(f"{'='*80}")
    summary = []
    for pname, sens_data in [('Threshold', th_sens), ('Long-Only', lo_sens), ('Volume Filter', vf_sens)]:
        if len(sens_data) > 0 and 'sharpe_delta' in sens_data.columns:
            summary.append({
                'Parameter': pname,
                'IS Sharpe Range': f"{sens_data['sharpe_ratio'].min():.3f} - {sens_data['sharpe_ratio'].max():.3f}",
                'Max Delta %': f"{sens_data['sharpe_delta'].min():.1f}%",
                'Sensitivity': 'HIGH' if abs(sens_data['sharpe_delta'].min()) > 40 else 'LOW',
            })
    print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════════
# MULTI-ASSET OOS — Does the academic finding generalize?
# ════════════════════════════════════════════════════════════════

if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    best_th = best['threshold']
    best_lo = bool(best['long_only'])
    best_vf = bool(best['volume_filter'])
    best_vm = best['vol_filter_mult']
    
    print(f"Best params from {PRIMARY_TICKER}: threshold={best_th}, long_only={best_lo}, "
          f"vol_filter={best_vf}, vol_mult={best_vm}")
    print(f"\nRunning across all {len(TICKERS)} tickers...\n")
    
    multi_results = []
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    fig.suptitle('Intraday Momentum — Multi-Asset OOS Test', fontsize=14, fontweight='bold')
    
    for ti, ticker in enumerate(TICKERS):
        try:
            t_hourly = all_data[ticker].copy()
            t_daily = build_daily_signals(t_hourly)
            
            # Use only OOS period (last 40%)
            t_split = int(len(t_daily) * TRAIN_RATIO)
            t_val_daily = t_daily.iloc[t_split:].copy()
            val_dates_t = set(t_val_daily.index.date)
            t_hourly_copy = t_hourly.copy()
            t_hourly_copy['trade_date'] = t_hourly_copy.index.date
            t_val_hourly = t_hourly_copy[t_hourly_copy['trade_date'].isin(val_dates_t)].drop(columns=['trade_date'])
            t_val_close = t_val_hourly['Close'].astype(float).copy()
            t_val_close.name = 'price'
            
            # Generate signals
            e, x, se, sx = generate_intraday_momentum_signals(
                t_val_hourly, t_val_daily, threshold=best_th, long_only=best_lo,
                volume_filter=best_vf, vol_filter_mult=best_vm)
            
            if best_lo:
                pf = vbt.Portfolio.from_signals(close=t_val_close, entries=e, exits=x,
                                                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
            else:
                pf = vbt.Portfolio.from_signals(close=t_val_close, entries=e, exits=x,
                                                short_entries=se, short_exits=sx,
                                                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
            
            def safe(fn, default=None):
                try: return float(fn())
                except: return default
            
            sharpe = safe(lambda: pf.sharpe_ratio(freq='h'))
            ret = safe(pf.total_return)
            dd = safe(pf.max_drawdown)
            n_trades = len(pf.trades)
            
            # Correlation for this asset
            t_corr = t_daily['morning_return'].corr(t_daily['last_hour_return'])
            
            multi_results.append({
                'ticker': ticker,
                'oos_sharpe': sharpe,
                'oos_return': ret,
                'oos_dd': dd,
                'oos_trades': n_trades,
                'morning_lh_corr': t_corr,
            })
            
            # Plot equity curve
            ax = axes.flat[ti]
            eq = pf.value()
            color = '#059669' if (ret or 0) > 0 else '#DC2626'
            ax.plot(eq.index, eq.values, color=color, linewidth=1.5)
            ax.set_title(f"{ticker}\nSharpe={sharpe:.3f} | Ret={ret:.2%}" if sharpe else ticker, fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
            
        except Exception as ex:
            print(f"  {ticker}: ERROR — {ex}")
            multi_results.append({'ticker': ticker, 'oos_sharpe': None, 'oos_return': None,
                                  'oos_dd': None, 'oos_trades': 0, 'morning_lh_corr': None})
    
    plt.tight_layout()
    plt.show()
    
    multi_df = pd.DataFrame(multi_results)
    print(f"\n{'='*90}")
    print(f"MULTI-ASSET OOS RESULTS (best params from {PRIMARY_TICKER})")
    print(f"{'='*90}")
    print(multi_df.to_string(index=False))
    
    # Summary stats
    valid = multi_df.dropna(subset=['oos_sharpe'])
    if len(valid) > 0:
        print(f"\nAverage OOS Sharpe across {len(valid)} assets: {valid['oos_sharpe'].mean():.4f}")
        print(f"Positive Sharpe: {(valid['oos_sharpe'] > 0).sum()} / {len(valid)}")
        print(f"Average morning-to-last-hour correlation: {valid['morning_lh_corr'].mean():.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FTMO MONTE CARLO SIMULATION
# ════════════════════════════════════════════════════════════════
# Aggregate hourly strategy returns to daily, then simulate FTMO paths

if results_df.empty:
    print("No results.")
else:
    # Rebuild full-sample portfolio with best params
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    best_th = best['threshold']
    best_lo = bool(best['long_only'])
    best_vf = bool(best['volume_filter'])
    best_vm = best['vol_filter_mult']
    
    e_full, x_full, se_full, sx_full = generate_intraday_momentum_signals(
        hourly_df, daily_signals, threshold=best_th, long_only=best_lo,
        volume_filter=best_vf, vol_filter_mult=best_vm)
    
    if best_lo:
        pf_full = vbt.Portfolio.from_signals(close=hourly_close, entries=e_full, exits=x_full,
                                             init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
    else:
        pf_full = vbt.Portfolio.from_signals(close=hourly_close, entries=e_full, exits=x_full,
                                             short_entries=se_full, short_exits=sx_full,
                                             init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='h')
    
    # Get hourly returns and aggregate to daily
    hourly_rets = pf_full.returns()
    daily_strat_rets = hourly_rets.groupby(hourly_rets.index.date).apply(
        lambda x: (1 + x).prod() - 1).values
    daily_strat_rets = daily_strat_rets[~np.isnan(daily_strat_rets)]
    
    print(f"Daily strategy returns: {len(daily_strat_rets)} days")
    print(f"Mean daily return: {np.mean(daily_strat_rets):.6f}")
    print(f"Std daily return:  {np.std(daily_strat_rets):.6f}")
    
    # FTMO Monte Carlo
    N_SIM = 10_000
    DAYS = 30
    ACCOUNT = 100_000
    TARGET = 0.10      # 10% profit target
    MAX_DAILY_DD = 0.05  # 5% max daily loss
    MAX_TOTAL_DD = 0.10  # 10% max total drawdown
    
    np.random.seed(42)
    n_passed = n_blown_daily = n_blown_total = 0
    final_eqs = np.zeros(N_SIM)
    sample_paths = []
    
    for s in range(N_SIM):
        eq = ACCOUNT
        path = [ACCOUNT]
        sim_rets = np.random.choice(daily_strat_rets, size=DAYS, replace=True)
        blown = False
        for d in range(DAYS):
            day_start = eq
            eq *= (1 + sim_rets[d])
            path.append(eq)
            # Check daily drawdown
            if (eq - day_start) / ACCOUNT < -MAX_DAILY_DD:
                n_blown_daily += 1; blown = True; break
            # Check total drawdown
            if (eq - ACCOUNT) / ACCOUNT < -MAX_TOTAL_DD:
                n_blown_total += 1; blown = True; break
            # Check profit target
            if (eq - ACCOUNT) / ACCOUNT >= TARGET:
                n_passed += 1; blown = True; break
        while len(path) < DAYS + 1:
            path.append(path[-1])
        final_eqs[s] = path[-1]
        if s < 200:
            sample_paths.append(path)
    
    n_still = N_SIM - n_passed - n_blown_total - n_blown_daily
    pass_rate = n_passed / N_SIM * 100
    
    verdict = ("FAVORABLE" if pass_rate >= 50 else "POSSIBLE" if pass_rate >= 25
               else "CHALLENGING" if pass_rate >= 10 else "UNLIKELY")
    
    print(f"\n{'='*70}")
    print(f"FTMO MONTE CARLO SIMULATION — {N_SIM:,} paths x {DAYS} days")
    print(f"{'='*70}")
    print(f"  PASSED (10% target):     {n_passed:>6,}  ({n_passed/N_SIM*100:.1f}%)")
    print(f"  BLOWN (daily DD):        {n_blown_daily:>6,}  ({n_blown_daily/N_SIM*100:.1f}%)")
    print(f"  BLOWN (total DD):        {n_blown_total:>6,}  ({n_blown_total/N_SIM*100:.1f}%)")
    print(f"  STILL RUNNING:           {n_still:>6,}  ({n_still/N_SIM*100:.1f}%)")
    print(f"\n  VERDICT: {verdict} (pass rate: {pass_rate:.1f}%)")
    
    # Plot Monte Carlo paths
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'FTMO Monte Carlo — {PRIMARY_TICKER} Intraday Momentum', fontsize=14, fontweight='bold')
    
    # Equity paths
    ax = axes[0]
    for path in sample_paths:
        color = '#059669' if path[-1] >= ACCOUNT * 1.10 else '#DC2626' if path[-1] < ACCOUNT * 0.90 else '#8C95A3'
        ax.plot(range(len(path)), path, color=color, alpha=0.08, linewidth=0.5)
    ax.axhline(ACCOUNT * 1.10, color='#059669', linestyle='--', linewidth=2, label='Target (+10%)')
    ax.axhline(ACCOUNT * 0.95, color='#EA580C', linestyle='--', linewidth=1.5, label='Daily DD (-5%)')
    ax.axhline(ACCOUNT * 0.90, color='#DC2626', linestyle='--', linewidth=2, label='Max DD (-10%)')
    ax.set_xlabel('Trading Day', fontsize=10)
    ax.set_ylabel('Account Value ($)', fontsize=10)
    ax.set_title('Simulated Equity Paths', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    
    # Final equity distribution
    ax = axes[1]
    ax.hist(final_eqs, bins=60, color='#2563EB', edgecolor='white', alpha=0.75)
    ax.axvline(ACCOUNT * 1.10, color='#059669', linestyle='--', linewidth=2, label='Target')
    ax.axvline(ACCOUNT * 0.90, color='#DC2626', linestyle='--', linewidth=2, label='Max DD')
    ax.axvline(np.median(final_eqs), color='#EA580C', linestyle='-', linewidth=2, label=f'Median: ${np.median(final_eqs):,.0f}')
    ax.set_xlabel('Final Account Value ($)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'Final Equity Distribution (Pass Rate: {pass_rate:.1f}%)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# UNIVERSAL STRATEGY EXPORT — Data Files + PDF Tearsheet
# ════════════════════════════════════════════════════════════════════
# INSTRUCTIONS:
#   1. Paste at the END of any strategy notebook
#   2. Edit STRATEGY_NAME and PARAM_COLS below
#   3. Run — exports structured data + a comprehensive PDF report
# ════════════════════════════════════════════════════════════════════

import os, sys, json, datetime, hashlib, platform
from matplotlib.backends.backend_pdf import PdfPages

# ═══ EDIT THESE LINES ═══════════════════════════════════════
STRATEGY_NAME = "Intraday_Momentum"                            # ← EDIT
PARAM_COLS    = ["threshold", "long_only", "volume_filter"]     # ← EDIT
NOTES         = "Gao, Han, Li & Zhou (JFE 2018) — first hour predicts last hour"  # ← Optional
# ════════════════════════════════════════════════════════════════

INIT_CASH = 100_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = 'h'  # hourly frequency for this strategy

# ── Google Drive mount ──
EXPORT_DIR = "/content/strategy_exports"
IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = "/content/drive/MyDrive/strategy_exports"
    IN_COLAB = True
    print("Google Drive mounted")
except:
    print("Local mode — exports to ./strategy_exports")

RUN_TIMESTAMP = datetime.datetime.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S")

# ── Folder structure ──
STRAT_DIR   = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
LATEST_DIR  = os.path.join(STRAT_DIR, "latest")
ARCHIVE_DIR = os.path.join(STRAT_DIR, "archive")
os.makedirs(LATEST_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════
# Build portfolios for export
# ════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
best_params = {
    'threshold': float(best['threshold']),
    'long_only': bool(best['long_only']),
    'volume_filter': bool(best['volume_filter']),
    'vol_filter_mult': float(best['vol_filter_mult']),
}
param_str = ", ".join([f"{k}={v}" for k, v in best_params.items()])

# Full-sample close (hourly)
full_close = hourly_close.copy()

# Full sample portfolio
e_full, x_full, se_full, sx_full = generate_intraday_momentum_signals(
    hourly_df, daily_signals, **best_params)
if best_params['long_only']:
    pf_full = vbt.Portfolio.from_signals(close=full_close, entries=e_full, exits=x_full,
                                         init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
else:
    pf_full = vbt.Portfolio.from_signals(close=full_close, entries=e_full, exits=x_full,
                                         short_entries=se_full, short_exits=sx_full,
                                         init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# IS portfolio
e_is, x_is, se_is, sx_is = generate_intraday_momentum_signals(
    train_hourly, train_daily, **best_params)
if best_params['long_only']:
    pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                       init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
else:
    pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                       short_entries=se_is, short_exits=sx_is,
                                       init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# OOS portfolio
e_oos, x_oos, se_oos, sx_oos = generate_intraday_momentum_signals(
    val_hourly, val_daily, **best_params)
if best_params['long_only']:
    pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
else:
    pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                        short_entries=se_oos, short_exits=sx_oos,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# Buy & Hold
bh_e = pd.Series(False, index=full_close.index, dtype=bool); bh_e.iloc[0] = True
bh_x = pd.Series(False, index=full_close.index, dtype=bool)
pf_bh = vbt.Portfolio.from_signals(close=full_close, entries=bh_e, exits=bh_x,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# ── Extract metrics ──
trades_obj = pf_full.trades
tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
pos, neg = tr[tr > 0], tr[tr < 0]
years_full = max((full_close.index[-1] - full_close.index[0]).days / 365.25, 1e-9)
daily_rets = pf_full.returns()

# Determine split index on hourly data
split_idx_h = len(train_close)

def safe(fn, default=None):
    try: return float(fn())
    except: return default

M = {
    'total_return': safe(pf_full.total_return), 'ann_return': safe(lambda: pf_full.annualized_return(freq=FREQ)),
    'sharpe': safe(lambda: pf_full.sharpe_ratio(freq=FREQ)), 'sortino': safe(lambda: pf_full.sortino_ratio(freq=FREQ)),
    'max_dd': safe(pf_full.max_drawdown), 'volatility': safe(lambda: pf_full.annualized_volatility(freq=FREQ)),
    'calmar': safe(lambda: pf_full.annualized_return(freq=FREQ)) / abs(safe(pf_full.max_drawdown)) if abs(safe(pf_full.max_drawdown, 0)) > 1e-9 else None,
    'trades': len(trades_obj), 'trades_yr': len(trades_obj) / years_full,
    'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else None,
    'pf': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else None,
    'expectancy': float(tr.mean()) if len(tr) > 0 else None,
    'avg_win': float(pos.mean()) if len(pos) > 0 else None, 'avg_loss': float(neg.mean()) if len(neg) > 0 else None,
    'largest_win': float(pos.max()) if len(pos) > 0 else None, 'largest_loss': float(neg.min()) if len(neg) > 0 else None,
    'payoff': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else None,
    'is_sharpe': safe(lambda: pf_is.sharpe_ratio(freq=FREQ)), 'is_return': safe(pf_is.total_return),
    'is_dd': safe(pf_is.max_drawdown), 'is_trades': len(pf_is.trades),
    'oos_sharpe': safe(lambda: pf_oos.sharpe_ratio(freq=FREQ)), 'oos_return': safe(pf_oos.total_return),
    'oos_dd': safe(pf_oos.max_drawdown), 'oos_trades': len(pf_oos.trades),
    'bh_return': safe(pf_bh.total_return), 'bh_sharpe': safe(lambda: pf_bh.sharpe_ratio(freq=FREQ)),
    'bh_dd': safe(pf_bh.max_drawdown),
}

# ════════════════════════════════════════════════════════════════
# 1. SAVE STRUCTURED DATA FILES
# ════════════════════════════════════════════════════════════════
export_json = {
    "metadata": {
        "run_id": RUN_ID, "export_timestamp": RUN_TIMESTAMP.isoformat(),
        "export_date_human": RUN_TIMESTAMP.strftime("%B %d, %Y at %I:%M %p"),
        "strategy_name": STRATEGY_NAME, "strategy_family": "Intraday_Momentum",
        "ticker": TICKER, "instrument_type": "equity/etf",
        "data_source": "alpaca", "data_interval": "1h", "currency": "USD",
        "start_date": str(full_close.index[0].date()), "end_date": str(full_close.index[-1].date()),
        "total_bars": len(full_close), "total_years": round(years_full, 2),
        "train_start": str(train_close.index[0].date()), "train_end": str(train_close.index[-1].date()),
        "train_bars": len(train_close), "val_start": str(val_close.index[0].date()),
        "val_end": str(val_close.index[-1].date()), "val_bars": len(val_close), "train_ratio": 0.60,
        "init_cash": INIT_CASH, "fees_pct": FEES, "slippage_pct": SLIPPAGE, "frequency": FREQ,
        "first_close": round(float(full_close.iloc[0]), 4), "last_close": round(float(full_close.iloc[-1]), 4),
        "python_version": sys.version.split()[0], "environment": "colab_pro" if IN_COLAB else "local",
        "grid_combos_tested": len(results_df), "param_columns": PARAM_COLS, "notes": NOTES,
        "academic_reference": "Gao, Han, Li & Zhou (2018). Market Intraday Momentum. Journal of Financial Economics.",
    },
    "best_params": best_params,
    "metrics_full_sample": {k: v for k, v in M.items() if not k.startswith('is_') and not k.startswith('oos_') and not k.startswith('bh_')},
    "metrics_in_sample": {k.replace('is_',''): v for k, v in M.items() if k.startswith('is_')},
    "metrics_out_of_sample": {k.replace('oos_',''): v for k, v in M.items() if k.startswith('oos_')},
    "metrics_buy_hold": {k.replace('bh_',''): v for k, v in M.items() if k.startswith('bh_')},
    "grid_search_summary": {
        "top5": results_df.nlargest(5, 'sharpe_ratio')[PARAM_COLS + ['sharpe_ratio','total_return','max_drawdown']].to_dict('records'),
    }
}

# Save JSON
with open(os.path.join(LATEST_DIR, "summary.json"), 'w', encoding='utf-8') as f:
    json.dump(export_json, f, indent=2, default=str)
with open(os.path.join(ARCHIVE_DIR, f"{RUN_ID}_summary.json"), 'w', encoding='utf-8') as f:
    json.dump(export_json, f, indent=2, default=str)
print(f"summary.json saved")

# Save CSVs
hourly_rets_export = pf_full.returns()
pd.DataFrame({'datetime': full_close.index.strftime('%Y-%m-%d %H:%M'), 'strategy_return': hourly_rets_export.values,
              'close': full_close.values, 'portfolio_value': pf_full.value().values
}).to_csv(os.path.join(LATEST_DIR, "hourly_returns.csv"), index=False)
print(f"hourly_returns.csv saved")

pd.DataFrame({'trade_num': range(1, len(tr)+1), 'return_pct': tr*100, 'pnl_usd': pnl,
              'cumulative_pnl': np.cumsum(pnl), 'is_winner': tr > 0
}).to_csv(os.path.join(LATEST_DIR, "trades.csv"), index=False)
print(f"trades.csv saved")

results_df.to_csv(os.path.join(LATEST_DIR, "grid_results.csv"), index=False)
print(f"grid_results.csv saved")

# Run log
log_path = os.path.join(EXPORT_DIR, "run_log.csv")
log_entry = pd.DataFrame([{
    "run_id": RUN_ID, "timestamp": RUN_TIMESTAMP.isoformat(), "strategy": STRATEGY_NAME,
    "ticker": TICKER, "best_params": str(best_params),
    "sharpe_full": round(M['sharpe'] or 0, 4), "sharpe_is": round(M['is_sharpe'] or 0, 4),
    "sharpe_oos": round(M['oos_sharpe'] or 0, 4), "total_return": round(M['total_return'] or 0, 4),
    "max_drawdown": round(M['max_dd'] or 0, 4), "total_trades": M['trades'],
    "win_rate": round(M['win_rate'] or 0, 1), "profit_factor": round(M['pf'] or 0, 2) if M['pf'] else None,
    "notes": NOTES, "export_path": STRAT_DIR,
}])
if os.path.exists(log_path):
    log_combined = pd.concat([pd.read_csv(log_path), log_entry], ignore_index=True)
else:
    log_combined = log_entry
log_combined.to_csv(log_path, index=False)
print(f"run_log.csv saved ({len(log_combined)} runs)")

# ════════════════════════════════════════════════════════════════
# 2. PDF TEARSHEET
# ════════════════════════════════════════════════════════════════
pdf_path = os.path.join(LATEST_DIR, "tearsheet.pdf")
pdf_archive = os.path.join(ARCHIVE_DIR, f"{RUN_ID}_tearsheet.pdf")

fmt = lambda v, d=2: f"{v:.{d}f}" if v is not None and not np.isnan(v) else "N/A"
fmtp = lambda v: f"{v:.2%}" if v is not None and not np.isnan(v) else "N/A"

BG       = '#FFFFFF'
CARD_BG  = '#F7F8FA'
CARD_BRD = '#E2E5EB'
TEXT_PRI = '#1A1D23'
TEXT_SEC = '#5A6270'
TEXT_MUT = '#8C95A3'
ACCENT   = '#2563EB'
GREEN    = '#059669'
RED      = '#DC2626'
ORANGE   = '#EA580C'
GRID_CLR = '#E5E7EB'

def _draw_card(ax_fig, x, y, w, h, label, value, color=ACCENT, fontsize_val=22):
    from matplotlib.patches import FancyBboxPatch
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008",
                           facecolor=CARD_BG, edgecolor=CARD_BRD, linewidth=1.2,
                           transform=ax_fig.transAxes, zorder=2)
    ax_fig.add_patch(rect)
    ax_fig.text(x + w/2, y + h*0.68, value, ha='center', va='center',
                fontsize=fontsize_val, fontweight='bold', color=color,
                transform=ax_fig.transAxes, zorder=3)
    ax_fig.text(x + w/2, y + h*0.25, label, ha='center', va='center',
                fontsize=8, color=TEXT_SEC, transform=ax_fig.transAxes, zorder=3)

def _style_ax(ax, title=None):
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT_SEC, labelsize=8)
    ax.grid(True, alpha=0.4, color=GRID_CLR, linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_color(CARD_BRD); spine.set_linewidth(0.8)
    if title:
        ax.set_title(title, color=TEXT_PRI, fontsize=11, fontweight='bold', pad=10)

with PdfPages(pdf_path) as pdf:
    # PAGE 1: Dashboard
    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    from matplotlib.patches import FancyBboxPatch, Rectangle
    header = Rectangle((0, 0.91), 1, 0.09, facecolor=ACCENT, transform=ax.transAxes, zorder=1)
    ax.add_patch(header)
    ax.text(0.04, 0.955, "STRATEGY TEARSHEET", ha='left', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes, zorder=2)
    ax.text(0.96, 0.955, f"{STRATEGY_NAME}  |  {TICKER}", ha='right', va='center',
            fontsize=12, color='rgba(255,255,255,0.85)', transform=ax.transAxes, zorder=2, family='monospace')
    ax.text(0.96, 0.92, f"{full_close.index[0].date()} to {full_close.index[-1].date()}  |  {len(full_close)} bars  |  {param_str}",
            ha='right', va='center', fontsize=8, color='rgba(255,255,255,0.65)', transform=ax.transAxes, zorder=2)
    card_w, card_h = 0.145, 0.09; card_y = 0.79
    cards = [
        ("SHARPE", fmt(M['sharpe'], 3), ACCENT),
        ("RETURN", fmtp(M['total_return']), GREEN if (M['total_return'] or 0) > 0 else RED),
        ("MAX DD", fmtp(M['max_dd']), RED if (M['max_dd'] or 0) < -0.15 else ORANGE),
        ("WIN RATE", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", GREEN if (M['win_rate'] or 0) > 50 else RED),
        ("TRADES", str(M['trades']), TEXT_PRI),
        ("PROFIT FACTOR", fmt(M['pf']), GREEN if (M['pf'] or 0) > 1.5 else (ORANGE if (M['pf'] or 0) > 1 else RED)),
    ]
    x_start = 0.03; gap = (0.94 - len(cards) * card_w) / (len(cards) - 1)
    for i, (label, value, color) in enumerate(cards):
        _draw_card(ax, x_start + i * (card_w + gap), card_y, card_w, card_h, label, value, color)
    rows = [
        ["METRIC", "FULL SAMPLE", "IN-SAMPLE", "OUT-OF-SAMPLE", "BUY & HOLD"],
        ["Total Return", fmtp(M['total_return']), fmtp(M['is_return']), fmtp(M['oos_return']), fmtp(M['bh_return'])],
        ["Sharpe Ratio", fmt(M['sharpe'], 3), fmt(M['is_sharpe'], 3), fmt(M['oos_sharpe'], 3), fmt(M['bh_sharpe'], 3)],
        ["Sortino Ratio", fmt(M['sortino'], 3), "--", "--", "--"],
        ["Max Drawdown", fmtp(M['max_dd']), fmtp(M['is_dd']), fmtp(M['oos_dd']), fmtp(M['bh_dd'])],
        ["Calmar Ratio", fmt(M['calmar'], 3), "--", "--", "--"],
        ["Volatility (Ann.)", fmtp(M['volatility']), "--", "--", "--"],
        ["Win Rate", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", "--", "--", "--"],
        ["Profit Factor", fmt(M['pf']), "--", "--", "--"],
        ["Expectancy", fmt(M['expectancy'], 4), "--", "--", "--"],
        ["Payoff Ratio", fmt(M['payoff']), "--", "--", "--"],
        ["Total Trades", str(M['trades']), str(M['is_trades']), str(M['oos_trades']), "1"],
        ["Trades/Year", fmt(M['trades_yr'], 1), "--", "--", "--"],
        ["Avg Win", fmtp(M['avg_win']), "--", "--", "--"],
        ["Avg Loss", fmtp(M['avg_loss']), "--", "--", "--"],
        ["Largest Win", fmtp(M['largest_win']), "--", "--", "--"],
        ["Largest Loss", fmtp(M['largest_loss']), "--", "--", "--"],
    ]
    table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
                     bbox=[0.03, 0.04, 0.94, 0.70])
    table.auto_set_font_size(False); table.set_fontsize(9)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(CARD_BRD); cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor(ACCENT); cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        else:
            cell.set_facecolor(BG if row % 2 == 1 else CARD_BG)
            cell.set_text_props(color=TEXT_PRI, fontsize=8, family='monospace')
            if col == 0: cell.set_text_props(color=TEXT_PRI, fontsize=8, fontweight='bold')
        cell.set_height(0.052)
    ax.text(0.5, 0.01, f"Run {RUN_ID}  |  QS Finance", ha='center', va='bottom',
            fontsize=7, color=TEXT_MUT, transform=ax.transAxes)
    pdf.savefig(fig, facecolor=BG); plt.close(fig)

    # PAGE 2: Equity + Drawdown
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={'height_ratios': [3, 1]})
    fig.patch.set_facecolor(BG)
    fig.suptitle(f'{STRATEGY_NAME} on {TICKER} — Equity & Drawdown', fontsize=14, fontweight='bold', color=TEXT_PRI, y=0.97)
    eq_strat = pf_full.value(); eq_bh = pf_bh.value()
    _style_ax(ax1)
    ax1.plot(full_close.index[:split_idx_h], eq_strat.iloc[:split_idx_h].values, color=ACCENT, linewidth=2, label='Strategy (IS)')
    ax1.plot(full_close.index[split_idx_h:], eq_strat.iloc[split_idx_h:].values, color=ORANGE, linewidth=2, label='Strategy (OOS)')
    ax1.plot(full_close.index, eq_bh.values, color=TEXT_MUT, linewidth=1.2, alpha=0.6, linestyle='--', label='Buy & Hold')
    ax1.axvline(x=full_close.index[split_idx_h], color=RED, linestyle=':', alpha=0.3, linewidth=1)
    ax1.set_ylabel('Portfolio Value ($)', color=TEXT_SEC, fontsize=9)
    ax1.legend(fontsize=8, facecolor=BG, edgecolor=CARD_BRD)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    stats_text = f"Sharpe: {fmt(M['sharpe'],3)}   |   Return: {fmtp(M['total_return'])}   |   MaxDD: {fmtp(M['max_dd'])}   |   WR: {M['win_rate']:.1f}%   |   PF: {fmt(M['pf'])}"
    ax1.text(0.5, 0.03, stats_text, transform=ax1.transAxes, fontsize=8, ha='center', color=TEXT_SEC, family='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))
    _style_ax(ax2)
    dd = pf_full.drawdown() * 100
    ax2.fill_between(full_close.index, dd.values, 0, color=RED, alpha=0.2)
    ax2.plot(full_close.index, dd.values, color=RED, linewidth=0.8, alpha=0.7)
    ax2.set_ylabel('Drawdown %', color=TEXT_SEC, fontsize=9)
    ax2.set_xlabel('Date', color=TEXT_SEC, fontsize=9)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig, facecolor=BG); plt.close(fig)

    # PAGE 3: Trade Analysis
    if len(tr) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
        fig.patch.set_facecolor(BG)
        fig.suptitle(f'Trade-by-Trade Analysis — {len(tr)} Trades', fontsize=14, fontweight='bold', color=TEXT_PRI, y=0.97)
        n = len(tr)
        colors_bar = [GREEN if r > 0 else RED for r in tr]
        colors_pnl = [GREEN if p > 0 else RED for p in pnl]
        for a in axes.flat: _style_ax(a)
        axes[0,0].bar(range(n), tr*100, color=colors_bar, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,0].axhline(np.mean(tr)*100, color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: {np.mean(tr)*100:.2f}%')
        axes[0,0].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[0,0].set_title('Trade Returns (%)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,0].legend(fontsize=7)
        axes[0,1].bar(range(n), pnl, color=colors_pnl, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,1].axhline(np.mean(pnl), color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: ${np.mean(pnl):,.0f}')
        axes[0,1].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[0,1].set_title('Trade P&L ($)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[0,1].legend(fontsize=7)
        cum_pnl = np.cumsum(pnl)
        axes[1,0].plot(range(1, n+1), cum_pnl, color=ACCENT, linewidth=2.5)
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl>=0, alpha=0.12, color=GREEN)
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl<0, alpha=0.12, color=RED)
        axes[1,0].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[1,0].set_title('Cumulative P&L', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[1,1].hist(tr*100, bins=min(30, max(10, n//3)), color=ACCENT, edgecolor='white', alpha=0.75)
        axes[1,1].axvline(np.mean(tr)*100, color=RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(tr)*100:.2f}%')
        axes[1,1].axvline(0, color=TEXT_MUT, linewidth=0.8, alpha=0.5)
        axes[1,1].set_title('Return Distribution', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,1].legend(fontsize=7)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        pdf.savefig(fig, facecolor=BG); plt.close(fig)

import shutil
shutil.copy2(pdf_path, pdf_archive)
print(f"tearsheet.pdf saved")

print(f"\n{'='*70}")
print(f"EXPORT COMPLETE — {STRATEGY_NAME} on {TICKER}")
print(f"{'='*70}")
print(f"  Run ID:     {RUN_ID}")
print(f"  Export dir: {STRAT_DIR}")
print(f"  Sharpe:     {fmt(M['sharpe'], 3)}")
print(f"  Return:     {fmtp(M['total_return'])}")
print(f"  Max DD:     {fmtp(M['max_dd'])}")
print(f"  Trades:     {M['trades']}")